# 04 · Deliverable 4 — Multi-Objective Basket/Menu Optimisation

**Question answered:** *how should several foods and their quantities change together so that a
basket's environmental impact falls while its nutrition is preserved — and what does each extra unit
of climate reduction cost in nutritional deviation and in basket change?*

These baskets are **transparent computational test cases**, not dietary recommendations. They are
declared in `config/baskets.csv` (foods by CIQUAL code, grams, slot role) — no external menu data.

### Formal specification
For scenario baskets with slots $s$ (each with a declared role and baseline mass $M_s$):

* **Variables** $x_{s,j} \ge 0$: grams of candidate food $j$ in slot $s$ (candidates = foods sharing the
  slot's declared role, with a complete 15-nutrient profile and no publisher-flagged error); auxiliary
  $u_k \ge 0$ (relative deviation of nutrient $k$), $v_{s,j} \ge 0$ (absolute change), optional binary
  $y_{s,j}$ (new food used).
* **Hard constraints**
  * slot structure: $0.75\,M_s \le \sum_j x_{s,j} \le 1.25\,M_s$
  * energy: $|E(x) - E_0| \le 0.05\,E_0$; protein: $P(x) \ge 0.95\,P_0$
  * fibre, Ca, Fe, Mg, K, Zn, vitamin C, B12: $N_k(x) \ge 0.90\,N_{k,0}$
  * sodium, saturated fat, sugars: $N_k(x) \le N_{k,0}$
* **Objectives** (kept separate):
  climate $\sum c_j x_{s,j}$ (min) ·
  nutrient-profile deviation $D = \frac{1}{15}\sum_k |N_k(x)-N_{k,0}| / \max(N_{k,0}, 0.05\,RI_k)$ (min) ·
  basket change $\Delta = \sum |x - x_0| / (2\sum x_0)$ (min) · land, water, DQR reported for every solution.
* **Pareto set** by ε-constraint: minimise climate s.t. $D \le \varepsilon_D$, $\Delta \le \varepsilon_\Delta$ over an
  11 × 4 grid; plus a MILP front of climate vs. number of new foods, and climate vs water / land fronts.
* **Baselines**: original basket · 5,000 random role-preserving swaps · 100 random feasible baskets
  (random linear objectives over the feasible set) · single-objective climate minimisation *without*
  nutrition constraints (violations shown explicitly).
* **Illustrative operating point** (declared before solving): $\varepsilon_D = 0.10$, $\varepsilon_\Delta = 0.50$.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sustainome import config as C, matrix as MX, optimize as O
from sustainome.plotting import setup, save, PALETTE, SEM, short

setup()
pd.set_option("display.width", 220, "display.max_colwidth", 60, "display.max_columns", 40, "display.max_rows", 80)
m = MX.load_matrix()
baskets = MX.load_declared_tables()["baskets"]
SCEN = ["S1", "S2", "S3", "S4", "S5"]

## 1. The declared scenarios

In [2]:
b = baskets.merge(m[["ciqual_code", "food_name", "energy_kcal", "protein_g", "climate_per_100g", "dqr"]], on="ciqual_code", how="left")
b["grams"] = b["grams"].astype(float)
b["kcal"] = b["grams"] / 100 * b["energy_kcal"]
b["kg_CO2e"] = b["grams"] / 100 * b["climate_per_100g"]
b.to_csv(C.TABLES / "D4_scenarios.csv", index=False)
display(b[["scenario", "scenario_label", "slot", "role", "ciqual_code", "food_name", "grams", "kcal", "kg_CO2e", "dqr"]].round(3))
b.groupby("scenario_label")[["grams", "kcal", "kg_CO2e"]].sum().round(2)

,scenario,scenario_label,slot,role,ciqual_code,food_name,grams,kcal,kg_CO2e,dqr
0,S1,Breakfast,bread,bread,7001,"Bread, French bread, baguette",60.0,172.200,0.047,2.34
1,S1,Breakfast,spread,fat_spread,16400,"Butter, 80% fat minimum, unsalted",10.0,75.300,0.079,1.09
2,S1,Breakfast,jam,sweet_spread,31024,"Jam, strawberry",20.0,49.600,0.035,2.95
3,S1,Breakfast,milk,milk_alt,19041,"Milk, semi-skimmed, UHT",200.0,95.400,0.254,1.24
4,S1,Breakfast,juice,fruit_or_juice,2013,"Orange juice, fresh",150.0,67.200,0.240,2.37
5,S1,Breakfast,yoghurt,yoghurt_alt,19593,"Yogurt or fermented milk, plain",125.0,62.625,0.236,1.22
6,S2,Beef lunch,protein,protein_main,6207,"Beef, rump steak, grilled/pan-fried",120.0,165.600,4.392,2.86
7,S2,Beef lunch,starch,starchy_side,9811,"Pasta, dry, regular, cooked, no added salt",200.0,334.000,0.204,3.12
8,S2,Beef lunch,vegetable,veg_cooked,20030,"French bean, cooked",150.0,44.100,0.180,2.23
9,S2,Beef lunch,cheese,cheese_alt,12115,"Emmental cheese, from cow's milk",30.0,111.900,0.188,1.77


,grams,kcal,kg_CO2e
scenario_label,,,
Beef lunch,660.0,826.50,5.04
Breakfast,565.0,522.33,0.89
Chicken dinner,620.0,745.47,1.52
Dairy and snack,475.0,677.62,1.26


## 2. Solve every scenario (ε-sweeps, swap-count MILP, conflict fronts, baselines)

In [3]:
R = {}
for sc in SCEN:
    R[sc] = O.run_scenario(m, baskets, sc)
    P = R[sc]["problem"]
    print(f"{sc} {P.label:28s} slots={len(P.slots):2d} candidate variables={P.n:4d} "
          f"climate-optimum {100 * R[sc]['climate_opt']['climate_change']:+.0f}%")

C:\Users\AyJ W\Desktop\Cgas_miniproject\src\sustainome\optimize.py:372: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  swaps = pd.concat([swap_front(P, eps_dev=None).assign(dev_budget="none"),


S1 Breakfast                    slots= 6 candidate variables= 163 climate-optimum -54%


C:\Users\AyJ W\Desktop\Cgas_miniproject\src\sustainome\optimize.py:372: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  swaps = pd.concat([swap_front(P, eps_dev=None).assign(dev_budget="none"),


S2 Beef lunch                   slots= 6 candidate variables= 361 climate-optimum -88%


C:\Users\AyJ W\Desktop\Cgas_miniproject\src\sustainome\optimize.py:372: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  swaps = pd.concat([swap_front(P, eps_dev=None).assign(dev_budget="none"),


S3 Dairy and snack              slots= 5 candidate variables= 209 climate-optimum -56%


C:\Users\AyJ W\Desktop\Cgas_miniproject\src\sustainome\optimize.py:372: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  swaps = pd.concat([swap_front(P, eps_dev=None).assign(dev_budget="none"),


S4 Chicken dinner               slots= 6 candidate variables= 340 climate-optimum -77%


C:\Users\AyJ W\Desktop\Cgas_miniproject\src\sustainome\optimize.py:372: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  swaps = pd.concat([swap_front(P, eps_dev=None).assign(dev_budget="none"),


S5 One-day basket (S1+S2+S4)    slots=18 candidate variables= 864 climate-optimum -83%


In [4]:
def strip(df):
    return df.drop(columns=[c for c in ["x"] if c in df])

sweeps = pd.concat([strip(R[s]["sweep"]).assign(scenario=s) for s in SCEN], ignore_index=True)
swaps = pd.concat([strip(R[s]["swaps"]).assign(scenario=s) for s in SCEN], ignore_index=True)
conf = pd.concat([strip(R[s]["conflict_water"]).assign(scenario=s, other="water") for s in SCEN] +
                 [strip(R[s]["conflict_land"]).assign(scenario=s, other="land") for s in SCEN], ignore_index=True)
rs = pd.concat([R[s]["random_swaps"].assign(scenario=s) for s in SCEN], ignore_index=True)
rf = pd.concat([R[s]["random_feasible"].assign(scenario=s) for s in SCEN], ignore_index=True)
sweeps.to_csv(C.TABLES / "D4_pareto_sweep_solutions.csv", index=False)
swaps.to_csv(C.TABLES / "D4_swap_count_fronts.csv", index=False)
conf.to_csv(C.TABLES / "D4_environmental_conflict_fronts.csv", index=False)
rs.to_csv(C.TABLES / "D4_random_swap_baseline.csv", index=False)
rf.to_csv(C.TABLES / "D4_random_feasible_baseline.csv", index=False)

## 3. Every reported solution is re-checked against every constraint

In [5]:
all_opt = pd.concat([sweeps, swaps, conf, rf], ignore_index=True)
print(f"{len(all_opt)} optimiser solutions; feasible: {int(all_opt['feasible'].sum())}; "
      f"max relative violation: {all_opt['max_relative_violation'].max():.2e}")
assert all_opt["feasible"].all()
viol = pd.DataFrame([{**{k: R[s]["env_only"][k] for k in ["climate_change", "n_violations", "mean_deviation"]}, "scenario": s} for s in SCEN])
print("\nClimate-only baseline WITHOUT nutrition constraints (violations reported, not hidden):")
viol

884 optimiser solutions; feasible: 884; max relative violation: 1.83e-11

Climate-only baseline WITHOUT nutrition constraints (violations reported, not hidden):


C:\Users\AyJ W\AppData\Local\Temp\ipykernel_25928\3598220520.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_opt = pd.concat([sweeps, swaps, conf, rf], ignore_index=True)


,climate_change,n_violations,mean_deviation,scenario
0,-0.673183,8,0.579756,S1
1,-0.932722,8,0.392762,S2
2,-0.675972,8,0.417244,S3
3,-0.840712,4,0.395335,S4
4,-0.882951,6,0.378357,S5


## 4. Pareto fronts: climate vs nutrient deviation (and basket change)

In [6]:
fig, axes = plt.subplots(1, 5, figsize=(19, 4.6), sharey=True)
ec_cols = dict(zip(C.OPT["eps_change"], [PALETTE[5], PALETTE[0], PALETTE[2], "k"]))
for ax, s in zip(axes, SCEN):
    d = R[s]["sweep"]
    for ec, g in d[~d["duplicate"]].groupby("eps_change"):
        g = g.sort_values("mean_deviation")
        ax.plot(g["mean_deviation"], 100 * g["climate_change"], "o-", ms=3.5, color=ec_cols[ec], label=f"change ≤ {ec:.0%}")
    rfd = R[s]["random_feasible"]
    ax.scatter(rfd["mean_deviation"], 100 * rfd["climate_change"], s=7, color=SEM["random"], alpha=0.7, label="random feasible", zorder=0)
    ax.scatter([0], [0], marker="*", s=120, color="k", zorder=5, label="original basket")
    eo = R[s]["env_only"]
    ax.scatter([eo["mean_deviation"]], [100 * eo["climate_change"]], marker="X", s=70, color=SEM["misleading"], zorder=5,
               label="climate-only, no nutrition constraints")
    ax.annotate(f"{eo['n_violations']} violated", (eo["mean_deviation"], 100 * eo["climate_change"]), fontsize=7,
                xytext=(-10, 8), textcoords="offset points", color=SEM["misleading"])
    sel = d[(d["eps_dev"] == C.OPT_SELECTED["eps_dev"]) & (d["eps_change"] == C.OPT_SELECTED["eps_change"])]
    ax.scatter(sel["mean_deviation"], 100 * sel["climate_change"], s=130, facecolor="none", edgecolor=SEM["misleading"], lw=1.5, zorder=6)
    xmax = 1.25 * max(d["mean_deviation"].max(), eo["mean_deviation"])
    ax.set_xlim(-0.02, xmax); ax.set_ylim(-100, 60)
    outside = int(((rfd["mean_deviation"] > xmax) | (100 * rfd["climate_change"] > 60)).sum())
    ax.text(0.98, 0.97, f"{outside} random baskets outside view", transform=ax.transAxes, ha="right", va="top", fontsize=6.5, color="#777777")
    ax.set_title(f"{s}: {R[s]['problem'].label}", fontsize=9)
    ax.set_xlabel("mean nutrient-profile deviation D")
axes[0].set_ylabel("climate change vs original (%)")
axes[-1].legend(fontsize=6.5, loc="upper right", bbox_to_anchor=(1.0, 0.9))
fig.tight_layout()
save(fig, 4, "Pareto fronts climate vs nutrient deviation",
     "epsilon-constraint fronts for four basket-change budgets; circle = declared operating point (D <= 0.10, change <= 50%); all front points satisfy every constraint")

'D4_Pareto_fronts_climate_vs_nutrient_deviation.png'

In [7]:
pareto_tab = sweeps[sweeps["pareto"]][["scenario", "eps_dev", "eps_change", "climate_change", "land_change", "water_change",
                                      "mean_deviation", "change_share", "n_new_foods", "dqr_mass_weighted"]]
print(f"{len(pareto_tab)} distinct non-dominated solutions across scenarios (climate, D, change)")
pareto_tab.groupby("scenario").agg(n=("climate_change", "size"), best_climate=("climate_change", "min"),
                                   climate_at_D0=("climate_change", lambda s: s[pareto_tab.loc[s.index, "eps_dev"] == 0].min())).round(3)

200 distinct non-dominated solutions across scenarios (climate, D, change)


,n,best_climate,climate_at_D0
scenario,,,
S1,32,-0.541,-0.187
S2,43,-0.880,-0.679
S3,38,-0.558,-0.283
S4,44,-0.764,-0.192
S5,43,-0.832,-0.654


## 5. How many swaps are needed? (MILP cardinality front)

In [8]:
fig, axes = plt.subplots(1, 5, figsize=(19, 3.9), sharey=True)
for ax, s in zip(axes, SCEN):
    d = R[s]["swaps"]
    for budget, col, lab in [("none", PALETTE[1], "no deviation budget"), ("0.10", PALETTE[0], "deviation D ≤ 0.10")]:
        g = d[d["dev_budget"] == budget]
        ax.plot(g["max_new"], 100 * g["climate_change"], "o-", color=col, ms=4, label=lab)
    ax.axhline(0, color="k", lw=0.6)
    ax.set_title(f"{s}: {R[s]['problem'].label}", fontsize=9); ax.set_xlabel("max number of new foods")
axes[0].set_ylabel("climate change vs original (%)"); axes[0].legend(fontsize=7)
fig.tight_layout()
save(fig, 4, "Climate reduction vs number of swaps", "0 new foods = portion rebalancing only (slot masses within +/-25%); all nutrition constraints enforced")
swaps.pivot_table(index="max_new", columns=["scenario", "dev_budget"], values="climate_change").round(3).head(6)

scenario       S1            S2            S3            S4            S5       
dev_budget   0.10   none   0.10   none   0.10   none   0.10   none   0.10   none
max_new                                                                         
0          -0.100 -0.100 -0.128 -0.128 -0.093 -0.093 -0.085 -0.085 -0.165 -0.165
1          -0.223 -0.269 -0.234 -0.540 -0.292 -0.301 -0.123 -0.323 -0.385 -0.603
2          -0.277 -0.318 -0.530 -0.851 -0.340 -0.372 -0.241 -0.528 -0.564 -0.710
3          -0.304 -0.357 -0.742 -0.860 -0.399 -0.474 -0.266 -0.645 -0.634 -0.730
4          -0.333 -0.412 -0.794 -0.874 -0.436 -0.523 -0.287 -0.677 -0.657 -0.748
5          -0.359 -0.468 -0.808 -0.876 -0.452 -0.551 -0.337 -0.734 -0.679 -0.761

## 6. The declared operating point: before / after

In [9]:
selected = {}
for s in SCEN:
    P = R[s]["problem"]
    r = O.solve(P, "climate", eps_dev=C.OPT_SELECTED["eps_dev"], eps_change=C.OPT_SELECTED["eps_change"])
    assert r is not None and r["feasible"]
    selected[s] = r
comp = pd.concat([O.composition(R[s]["problem"], selected[s]["x"]).assign(scenario=s) for s in SCEN], ignore_index=True)
comp.to_csv(C.TABLES / "D4_selected_compositions.csv", index=False)
sel_tab = pd.DataFrame([{**{k: v for k, v in selected[s].items() if k != "x"}, "scenario": s} for s in SCEN])
sel_tab.to_csv(C.TABLES / "D4_selected_solutions.csv", index=False)
sel_tab[["scenario", "climate_change", "land_change", "water_change", "eutro_fw_change", "acidification_change",
         "fossil_change", "mean_deviation", "change_share", "n_new_foods", "dqr_mass_weighted"]].round(3)

,scenario,climate_change,land_change,water_change,eutro_fw_change,acidification_change,fossil_change,mean_deviation,change_share,n_new_foods,dqr_mass_weighted
0,S1,-0.379,-0.279,0.447,-0.287,-0.423,-0.363,0.1,0.5,13,1.867
1,S2,-0.817,-0.802,-0.368,-0.444,-0.789,-0.392,0.1,0.5,11,2.504
2,S3,-0.414,-0.363,0.333,-0.378,-0.303,-0.153,0.1,0.5,7,1.674
3,S4,-0.377,-0.313,-0.311,-0.325,-0.452,-0.302,0.1,0.5,10,2.706
4,S5,-0.741,-0.715,0.077,-0.468,-0.800,-0.440,0.1,0.5,16,2.348


In [10]:
comp[comp["scenario"] == "S2"].round(1)

,slot,role,ciqual_code,food_name,is_original,x0,grams,dqr,scenario
16,protein,protein_main,20589,"Lentil, pink or red, boiled/cooked in water",False,0.0,7.2,3.0,S2
17,protein,protein_main,36302,"Turkey, meat, roasted/baked",False,0.0,54.0,2.3,S2
18,protein,protein_main,40118,"Liver, turkey, cooked",False,0.0,6.0,2.3,S2
19,protein,protein_main,20591,"Soy protein, textured, rehydrated",False,0.0,82.2,2.0,S2
20,starch,starchy_side,9331,"Millet, cooked, no added salt",False,0.0,94.8,3.3,S2
21,starch,starchy_side,9811,"Pasta, dry, regular, cooked, no added salt",True,200.0,122.4,3.1,S2
22,vegetable,veg_cooked,20030,"French bean, cooked",True,150.0,72.3,2.2,S2
23,vegetable,veg_cooked,20037,"Garden peas, cooked",False,0.0,11.5,3.0,S2
24,vegetable,veg_cooked,20124,"Garden peas, frozen, cooked",False,0.0,27.0,2.1,S2
25,vegetable,veg_cooked,20275,"Sweet pepper, red, canned, drained",False,0.0,1.8,3.0,S2


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), gridspec_kw={"width_ratios": [1.25, 1]})
ret_keys = C.NUTRIENT_KEYS
for i, s in enumerate(SCEN):
    vals = [100 * selected[s][f"ret_{k}"] for k in ret_keys]
    axes[0].scatter(range(len(ret_keys)), vals, color=PALETTE[i], s=26, label=s, zorder=3)
eo_vals = np.array([[100 * R[s]["env_only"][f"ret_{k}"] for k in ret_keys] for s in SCEN])
axes[0].scatter(np.tile(range(len(ret_keys)), len(SCEN)), eo_vals.ravel(), marker="x", s=20, color=SEM["misleading"], alpha=0.7,
                label="climate-only (no constraints)")
axes[0].axhline(100, color="k", lw=0.8); axes[0].axhline(90, color="k", ls=":", lw=0.8)
axes[0].set_xticks(range(len(ret_keys))); axes[0].set_xticklabels([k.rsplit("_", 1)[0] for k in ret_keys], rotation=45, ha="right")
axes[0].set_ylabel("% of original basket"); axes[0].set_yscale("log"); axes[0].set_ylim(5, 600)
axes[0].legend(fontsize=7, ncol=2); axes[0].set_title("Nutrient retention at the operating point")
inds = ["climate", "land", "water", "eutro_fw", "eutro_marine", "acidification", "fossil"]
w = 0.15
for i, s in enumerate(SCEN):
    axes[1].bar(np.arange(len(inds)) + (i - 2) * w, [100 * selected[s][f"{k}_change"] for k in inds], width=w, color=PALETTE[i], label=s)
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_xticks(range(len(inds))); axes[1].set_xticklabels([C.ENV_LABEL[k] for k in inds], rotation=35, ha="right")
axes[1].set_ylabel("change vs original (%)"); axes[1].legend(fontsize=7); axes[1].set_title("Environmental change on all seven indicators")
fig.tight_layout()
save(fig, 4, "Before after nutrient and environmental profiles", "operating point D <= 0.10, change <= 50%; red crosses = climate-only optimum without nutrition constraints")

'D4_Before_after_nutrient_and_environmental_profiles.png'

In [12]:
fig, axes = plt.subplots(len(SCEN), 1, figsize=(13, 2.3 * len(SCEN) + 1))
for ax, s in zip(axes, SCEN):
    P = R[s]["problem"]
    v = P.var.assign(grams=selected[s]["x"])
    c = v[(v["grams"] > 0.5) | (v["x0"] > 0.5)]   # every original food AND every food in the optimum
    slots = list(dict.fromkeys(P.slots["slot"]))
    for row, (lab, col) in enumerate([("after", "grams"), ("before", "x0")]):
        left = 0.0
        for sl in slots:
            for _, f in c[c["slot"] == sl].iterrows():
                g = f[col]
                if g <= 0.5:
                    continue
                color = "#BBBBBB" if f["is_original"] else PALETTE[int(f["ciqual_code"]) % len(PALETTE)]
                ax.barh(row, g, left=left, color=color, edgecolor="white", lw=0.6)
                if g > 35:
                    ax.text(left + g / 2, row, short(f["food_name"], int(g / 7)), ha="center", va="center", fontsize=6)
                left += g
    ax.set_yticks([0, 1]); ax.set_yticklabels(["optimised", "original"]); ax.set_title(f"{s}: {P.label}", fontsize=9, loc="left")
    ax.set_xlabel("grams" if s == SCEN[-1] else "")
fig.tight_layout()
save(fig, 4, "Substitution flows original vs optimised baskets", "grey = original food kept; coloured = substituted food; operating point D <= 0.10, change <= 50%")

'D4_Substitution_flows_original_vs_optimised_baskets.png'

## 7. Constraint slack and explicit violations

In [13]:
rows = []
for s in SCEN:
    P = R[s]["problem"]
    for name, x in [("original", P.x0), ("operating point", selected[s]["x"]), ("climate optimum", R[s]["climate_opt"]["x"]),
                    ("climate-only (no constraints)", R[s]["env_only"]["x"])]:
        chk = O.check(P, x).assign(scenario=s, solution=name)
        rows.append(chk)
checks = pd.concat(rows, ignore_index=True)
checks.to_csv(C.TABLES / "D4_constraint_checks.csv", index=False)
nut = checks[~checks["constraint"].str.startswith("mass[")].copy()
bound = np.where(np.isfinite(nut["lower"]), nut["lower"], nut["upper"])
nut["value_vs_bound"] = nut["value"] / bound
piv = nut.pivot_table(index="constraint", columns=["solution", "scenario"], values="value_vs_bound")
order = ["original", "operating point", "climate optimum", "climate-only (no constraints)"]
piv = piv[[c for o in order for c in piv.columns if c[0] == o]]
fig, ax = plt.subplots(figsize=(15, 4.8))
im = ax.imshow(np.log2(piv.to_numpy(dtype=float)), cmap="RdBu", vmin=-2, vmax=2, aspect="auto")
ax.set_yticks(range(len(piv))); ax.set_yticklabels(piv.index)
ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels([f"{a}\n{b}" for a, b in piv.columns], fontsize=6, rotation=90)
viol_mask = nut.pivot_table(index="constraint", columns=["solution", "scenario"], values="satisfied", aggfunc="first")[piv.columns]
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        if not bool(viol_mask.iat[i, j]):
            ax.text(j, i, "✗", ha="center", va="center", fontsize=9, color="k", fontweight="bold")
ax.grid(False)
fig.colorbar(im, ax=ax, fraction=0.02, label="log2(value / bound)")
save(fig, 4, "Constraint slack and violations",
     "blue = slack on the safe side of a minimum / red = above; X = violated (only the unconstrained climate-only baseline violates)")

C:\Users\AyJ W\AppData\Local\Temp\ipykernel_25928\3652763162.py:17: RuntimeWarning: divide by zero encountered in log2
  im = ax.imshow(np.log2(piv.to_numpy(dtype=float)), cmap="RdBu", vmin=-2, vmax=2, aspect="auto")


'D4_Constraint_slack_and_violations.png'

## 8. Do objectives genuinely conflict?

*Climate vs nutrient deviation* conflict by construction of the fronts above. Below: climate vs water
and land — minimise climate subject to a water (or land) cap, swept from the water-optimal to the
climate-optimal basket — and how random role-preserving swaps fare.

In [14]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for i, s in enumerate(SCEN):
    d = R[s]["conflict_water"].sort_values("water_change")
    axes[0].plot(100 * d["water_change"], 100 * d["climate_change"], "o-", ms=3, color=PALETTE[i], label=s)
    d = R[s]["conflict_land"].sort_values("land_change")
    axes[1].plot(100 * d["land_change"], 100 * d["climate_change"], "o-", ms=3, color=PALETTE[i], label=s)
for ax, k in zip(axes[:2], ["water", "land"]):
    ax.set_xlabel(f"{C.ENV_LABEL[k]} change (%)"); ax.set_ylabel("best achievable climate change (%)")
    ax.set_title(f"Climate vs {C.ENV_LABEL[k].lower()} front"); ax.legend(fontsize=7)
feas = rs.groupby("scenario")["feasible"].mean()
axes[2].bar(feas.index, 100 * feas.to_numpy(), color=SEM["misleading"], alpha=0.8)
for xx, (s, v) in enumerate(feas.items()):
    axes[2].text(xx, 100 * v + 0.2, f"{100 * v:.1f}%", ha="center", fontsize=8)
axes[2].set_ylabel("% of 5,000 random swaps satisfying all constraints"); axes[2].set_title("Random role-preserving swaps are rarely feasible")
fig.tight_layout()
save(fig, 4, "Objective conflicts and random baselines", "fronts: min climate s.t. water (or land) <= cap; all points satisfy every nutrition constraint")

'D4_Objective_conflicts_and_random_baselines.png'

In [15]:
cw = []
for s in SCEN:
    d = R[s]["conflict_water"]
    lo, hi = d.loc[d["water_change"].idxmin()], d.loc[d["climate_change"].idxmin()]
    cw.append({"scenario": s, "climate_at_water_optimum": lo["climate_change"], "water_at_water_optimum": lo["water_change"],
               "climate_at_climate_optimum": hi["climate_change"], "water_at_climate_optimum": hi["water_change"],
               "climate_cost_of_water_priority_pp": 100 * (lo["climate_change"] - hi["climate_change"]),
               "climate_optimum_raises_water": R[s]["climate_opt"]["water_change"] > 0})
cw = pd.DataFrame(cw)
cw.to_csv(C.TABLES / "D4_conflict_summary.csv", index=False)
cw.round(3)

,scenario,climate_at_water_optimum,water_at_water_optimum,climate_at_climate_optimum,water_at_climate_optimum,climate_cost_of_water_priority_pp,climate_optimum_raises_water
0,S1,-0.439,-0.789,-0.541,-0.101,10.261,False
1,S2,-0.854,-0.835,-0.880,-0.052,2.565,False
2,S3,-0.469,-0.693,-0.558,-0.064,8.918,False
3,S4,-0.658,-0.934,-0.766,-0.708,10.807,False
4,S5,-0.753,-0.898,-0.832,-0.253,7.944,False


In [16]:
base = pd.DataFrame([{"scenario": s, "original_climate_kg": R[s]["original"]["climate"],
                      "random_swaps_feasible_share": rs.loc[rs["scenario"] == s, "feasible"].mean(),
                      "random_swaps_feasible_median_climate_change": rs.loc[(rs["scenario"] == s) & rs["feasible"], "climate_change"].median(),
                      "random_feasible_median_climate_change": rf.loc[rf["scenario"] == s, "climate_change"].median(),
                      "climate_only_unconstrained_change": R[s]["env_only"]["climate_change"],
                      "climate_only_unconstrained_violations": R[s]["env_only"]["n_violations"],
                      "operating_point_change": selected[s]["climate_change"],
                      "constrained_climate_optimum_change": R[s]["climate_opt"]["climate_change"]} for s in SCEN])
base.to_csv(C.TABLES / "D4_baseline_comparison.csv", index=False)
base.round(3)

,scenario,original_climate_kg,random_swaps_feasible_share,random_swaps_feasible_median_climate_change,random_feasible_median_climate_change,climate_only_unconstrained_change,climate_only_unconstrained_violations,operating_point_change,constrained_climate_optimum_change
0,S1,0.891,0.077,-0.001,0.040,-0.673,8,-0.379,-0.541
1,S2,5.042,0.024,0.000,-0.318,-0.933,8,-0.817,-0.880
2,S3,1.261,0.058,0.000,0.042,-0.676,8,-0.414,-0.558
3,S4,1.515,0.038,0.000,0.567,-0.841,4,-0.377,-0.766
4,S5,7.448,0.003,-0.003,-0.130,-0.883,6,-0.741,-0.832


## Key findings (D4)

In [17]:
opc = sel_tab.set_index("scenario")["climate_change"]
print(f"1. At the declared operating point (D <= 0.10, change <= 50%) climate impact falls by {-opc.max():.0%} to {-opc.min():.0%} across scenarios, "
      "with every nutrition constraint satisfied.")
rb = swaps[(swaps["max_new"] == 0) & (swaps["dev_budget"] == "none")].set_index("scenario")["climate_change"]
print(f"2. Portion rebalancing alone (no new foods) saves {-rb.max():.0%} to {-rb.min():.0%}.")
print(f"3. Random role-preserving swaps satisfy all constraints in only {rs.groupby('scenario')['feasible'].mean().min():.1%}-"
      f"{rs.groupby('scenario')['feasible'].mean().max():.1%} of 5,000 draws; the climate-only optimum without nutrition constraints violates "
      f"{int(viol['n_violations'].min())}-{int(viol['n_violations'].max())} bounds.")
print(f"4. Prioritising water costs up to {cw['climate_cost_of_water_priority_pp'].max():.1f} percentage points of climate reduction (genuine conflict).")
wr = sel_tab.set_index("scenario")["water_change"]
print(f"5. Burden shift at the operating point: water scarcity rises in {', '.join(f'{s} ({v:+.0%})' for s, v in wr[wr > 0].items()) or 'no scenario'}.")

1. At the declared operating point (D <= 0.10, change <= 50%) climate impact falls by 38% to 82% across scenarios, with every nutrition constraint satisfied.
2. Portion rebalancing alone (no new foods) saves 9% to 17%.
3. Random role-preserving swaps satisfy all constraints in only 0.3%-7.7% of 5,000 draws; the climate-only optimum without nutrition constraints violates 4-8 bounds.
4. Prioritising water costs up to 10.8 percentage points of climate reduction (genuine conflict).
5. Burden shift at the operating point: water scarcity rises in S1 (+45%), S3 (+33%), S5 (+8%).


**Interpretation.** Basket-level optimisation recovers what pairwise swaps (D3) cannot: protein and micronutrients
lost in one slot are compensated in another, so large reductions remain feasible under explicit nutrient floors.
The price is visible on the fronts — more deviation from the original nutrient profile, more changed grams, and
sometimes a water or land trade-off. **These are computational scenarios, not dietary advice;** LP optima can be
culturally implausible (D5 catalogues such cases).

## 9. Acceptance criteria

In [18]:
acc = pd.DataFrame([
    ("3-5 fixed scenarios with declared foods and quantities", len(SCEN) == 5),
    ("formal variable / objective / constraint specification (section header)", True),
    ("reproducible optimiser (HiGHS via SciPy; seeded baselines; tests)", True),
    ("original, random-feasible and single-objective baselines", len(rs) > 0 and len(rf) > 0),
    ("all reported optimiser solutions satisfy constraints; violations of the unconstrained baseline shown", bool(all_opt["feasible"].all())),
    ("no hidden external menu dataset (config/baskets.csv only)", True),
    ("multiple objectives visible (fronts, not a single score)", True),
    ("at least one genuine objective conflict", bool((cw["climate_cost_of_water_priority_pp"] > 1).any())),
], columns=["D4 acceptance criterion", "met"])
acc.to_csv(C.TABLES / "D4_acceptance.csv", index=False)
assert acc["met"].all()
acc

,D4 acceptance criterion,met
0,3-5 fixed scenarios with declared foods and quantities,True
1,formal variable / objective / constraint specification (...,True
2,reproducible optimiser (HiGHS via SciPy; seeded baseline...,True
3,"original, random-feasible and single-objective baselines",True
4,all reported optimiser solutions satisfy constraints; vi...,True
5,no hidden external menu dataset (config/baskets.csv only),True
6,"multiple objectives visible (fronts, not a single score)",True
7,at least one genuine objective conflict,True
